# Digital Professor: Recognition Demo

This notebook tests the same recognition contracts that a later browser UI and FastAPI backend can call. API-backed cells remain optional until `OPENAI_API_KEY` is set in the root `.env`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from digital_professor import (
    RecognitionService,
    ingest_document,
    load_settings,
    render_document,
)
from digital_professor.equations import extract_delimited_equations

TEST_DATA = ROOT / 'resources' / 'interaction_test'
settings = load_settings(ROOT / '.env')
print(f'Project root: {ROOT}')
print(f'Model: {settings.openai_model}; API enabled: {settings.api_enabled}')

## Part 1 — Text/equation ingestion and generation

First inspect deterministic embedded-text extraction. This step makes no API call and keeps page provenance.

In [ ]:
notes_path = TEST_DATA / 'samp_notes.pdf'
notes = ingest_document(notes_path, max_pages=3)
print(f'{len(notes.pages)} page(s), {len(notes.combined_text):,} extracted characters')
display(Markdown(f'```text\n{notes.combined_text[:3000]}\n```'))

The local equation helper only claims expressions that already have TeX delimiters. PDF glyph recovery is left to visual/model recognition because ordinary text extraction often loses equation layout.

In [ ]:
typed_example = r'''For $f(x)=x^2$, the derivative is $f'(x)=2x$.
The normal equation is $$\hat{\beta}=(X^T X)^{-1}X^T y.$$'''
equations = extract_delimited_equations(typed_example)
display(Markdown(typed_example))
[item.model_dump() for item in equations]

### Model-assisted normalization and answer generation

This converts extracted PDF text to structured Markdown/LaTeX, then asks one grounded question. It runs only when a key is available.

In [ ]:
if settings.api_enabled:
    service = RecognitionService(settings)
    typed_result = service.recognize_embedded_text(notes_path)
    for page in typed_result.pages:
        display(Markdown(f'### Page {page.page_number}\n{page.markdown}'))
        if page.uncertainties:
            print('Review:', page.uncertainties)
else:
    typed_result = None
    print('Skipped: add OPENAI_API_KEY to .env, restart the kernel, and rerun.')

In [ ]:
if typed_result is not None:
    context = '\n\n'.join(page.markdown for page in typed_result.pages)
    generated = service.generate(
        'Choose one equation from these notes, explain every symbol, and give a short check-for-understanding question.',
        context,
    )
    display(Markdown(generated.answer_markdown))

## Part 2 — Handwritten recognition

Render the handwriting locally before sending it to a vision-capable model. This lets us verify exactly what the recognizer receives.

In [ ]:
handwriting_path = TEST_DATA / 'samp_solution.pdf'
preview = render_document(handwriting_path, dpi=settings.render_dpi, max_pages=1)[0]
display(Image(data=preview.image_bytes, width=850))

In [ ]:
if settings.api_enabled:
    service = RecognitionService(settings)
    handwritten_result = service.recognize_handwriting(handwriting_path, pages=[1])
    page = handwritten_result.pages[0]
    display(Markdown(page.markdown))
    display({
        'equations': [equation.model_dump() for equation in page.equations],
        'uncertainties': page.uncertainties,
    })
else:
    handwritten_result = None
    print('Skipped: add OPENAI_API_KEY to .env, restart the kernel, and rerun.')

### Compare easy, incorrect, and ambiguous samples

The recognizer is instructed to preserve mathematical errors and surface uncertain symbols instead of silently correcting them.

In [ ]:
comparison_files = [
    TEST_DATA / 'samp_solution.pdf',
    TEST_DATA / 'samp_wrong_solution.pdf',
    TEST_DATA / 'samp_bad_writing.pdf',
]

if settings.api_enabled:
    comparison = {}
    for path in comparison_files:
        result = service.recognize_handwriting(path, pages=[1])
        comparison[path.name] = result.model_dump()
        display(Markdown(f'### {path.name}\n{result.pages[0].markdown}'))
        print('Uncertainties:', result.pages[0].uncertainties)
else:
    print('Skipped API comparison.')